In [ ]:
from pathlib import Path
import sys
import importlib.util
import torch
import os
PROJECT_ROOT = Path.cwd()
# train.py 위치 탐색
train_py_candidates = [
    PROJECT_ROOT / "train.py",
    PROJECT_ROOT / "scripts" / "train.py",
]

TRAIN_PY = None
for path in train_py_candidates:
    if path.exists():
        TRAIN_PY = path
        break

if TRAIN_PY is None:
    raise FileNotFoundError(
        f"train.py를 찾지 못했습니다. PROJECT_ROOT를 수정하세요. 현재 PROJECT_ROOT={PROJECT_ROOT}"
    )

# import 경로 보정
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(TRAIN_PY.parent))
sys.path.insert(0, str(TRAIN_PY.parent.parent))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PY:", TRAIN_PY)
print("CUDA available:", torch.cuda.is_available())

/content
['.config', 'sample_data']
/content/scripts/train.py


FileNotFoundError: train.py를 찾지 못했습니다. PROJECT_ROOT를 수정하세요. 현재 PROJECT_ROOT=/content

In [ ]:
spec = importlib.util.spec_from_file_location("train_module", TRAIN_PY)
train_module = importlib.util.module_from_spec(spec)

# dataclass, checkpoint 호환성 보정
sys.modules["train_module"] = train_module

if spec.loader is None:
    raise ImportError(f"Cannot load module from {TRAIN_PY}")

spec.loader.exec_module(train_module)

print("Available models:", train_module.MODEL_NAMES)

def make_config(
    model_name: str = "resnet",
    dataset: str = "fashion",
    epochs: int = 5,
    batch_size: int = 128,
    learning_rate: float = 1e-3,
    weight_decay: float = 1e-3,
    optimizer: str = "adam",
    scheduler: str = "cosine",
    monitor: str = "valid_f1",
    monitor_mode: str = "max",
    train_ratio: float = 0.8,
    num_workers: int = 2,
    checkpoint_interval: int = 5,
    storage_root: str = "./storage",
    run_name: str | None = None,
    model_config_path: str | None = None,
    resume: str | None = None,
    seed: int = 42,
    gpu_id: int | None = None,
    verbose: int = 1,
):
    if gpu_id is None:
        gpu_id = 0 if torch.cuda.is_available() else -1

    args = [
        "--model_name", model_name,
        "--dataset", dataset,
        "--epochs", str(epochs),
        "--batch_size", str(batch_size),
        "--learning_rate", str(learning_rate),
        "--weight_decay", str(weight_decay),
        "--optimizer", optimizer,
        "--scheduler", scheduler,
        "--monitor", monitor,
        "--monitor_mode", monitor_mode,
        "--train_ratio", str(train_ratio),
        "--num_workers", str(num_workers),
        "--checkpoint_interval", str(checkpoint_interval),
        "--storage_root", storage_root,
        "--seed", str(seed),
        "--gpu_id", str(gpu_id),
        "--verbose", str(verbose),
    ]

    if run_name is not None:
        args.extend(["--run_name", run_name])

    if model_config_path is not None:
        args.extend(["--model_config_path", model_config_path])

    if resume is not None:
        args.extend(["--resume", resume])

    return train_module.parse_config(args)

In [ ]:
config = make_config(
    model_name="mlp",
    dataset="fashion",
    epochs=1,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-3,
    storage_root="./storage",
    run_name="smoke_fashion_mlp",
    model_config_path=None,
    verbose=1,
)

metrics = train_module.main(config)
metrics